[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc5_abtest/cours/seance1_cours.ipynb)

# Séance 5.1 — A/B testing — causalité et expériences randomisées

**Cours** · durée : 6h (2h de cours, 2h d'étude de cas, 2h de correction)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer une question prédictive d'une question causale
- raisonner avec les résultats potentiels, le contrefactuel, l'ATE et l'ATT
- décomposer une comparaison de moyennes en effet causal et biais de sélection
- expliquer ce que la randomisation change, et vérifier qu'elle a fonctionné
- mesurer l'effet d'une campagne, son incertitude, et en tirer une décision chiffrée

# 01 - Introduction à la causalité

À chaque vague d'IA, la même annonce : les data scientists vont disparaître, la machine analyse mieux qu'eux. On l'a dit à l'arrivée du Machine Learning, on le redit avec les LLM. Ce cours parie sur l'inverse : plus les outils deviennent puissants, plus il est précieux de savoir ce qu'on peut leur faire dire, et surtout ce qu'on ne peut pas...

![fmoa3nn4njue1.jpeg](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/fmoa3nn4njue1.jpeg)

Au milieu de ce battage médiatique, on oublie facilement que des gens font de la « science avec des données » depuis plus d'un siècle : statisticiens, économistes, épidémiologistes. La data science n'est pas une discipline nouvelle, elle bénéficie simplement, depuis quelques années, d'une publicité massive.

Jim Collins propose une image pour y voir clair : versez-vous une bière. Il y a la mousse: le battage, les promesses gonflées, les outils à la mode qui auront disparu dans cinq ans. Et il y a la bière: les fondations statistiques, la rigueur scientifique, le goût des problèmes difficiles. La mousse retombe toujours ; la bière, elle, a fait ses preuves depuis des siècles.

Ce cours est sur la bière : les fondamentaux statistiques sur lesquels repose toute la science des données. Les mathématiques et la statistique sont utiles depuis toujours, et il est peu probable que cela s'arrête demain. Apprenez ce qui rend votre travail réellement précieux, pas le dernier outil brillant que personne n'a encore compris.

## Quelle différence entre l'inférence causale et les questions classiques de machine learning ?

Les méthodes ne répondent pas au même type de question.

Machine Learning:

**Le machine learning** est très bon pour répondre à des questions de **type prédictif**. Comme l'ont dit Ajay Agrawal, Joshua Gans et Avi Goldfarb dans le livre Prediction Machines, **"la nouvelle vague d'intelligence artificielle ne nous apporte pas vraiment l'intelligence mais plutôt un composant critique de l'intelligence - la prédiction"**.

*Exemples:*

Vous voulez reconnaître des visages ? Alors créez un modèle ML qui prédit la présence d'un visage dans une sous-section d'une image. Vous voulez construire une voiture autonome ? Alors faites un modèle ML pour prédire la direction du volant et la pression sur les freins et l'accélérateur lorsqu'on lui présente des images et des capteurs provenant des environs d'une voiture.

*Limites:*

Cependant, le ML n'est pas une panacée. Il peut accomplir des merveilles dans des limites rigides et échouer lamentablement si ses données dévient un peu de ce à quoi le modèle est habitué. Pour donner un autre exemple tiré de Prediction Machines, "dans de nombreuses industries, les bas prix sont associés à de faibles ventes. Par exemple, dans l'industrie hôtelière, les prix sont bas hors saison touristique, et les prix sont élevés lorsque la demande est la plus forte et que les hôtels sont complets. Étant donné ces données, une prédiction naïve pourrait suggérer qu'augmenter le prix conduirait à vendre plus de chambres."

**Le ML est notoirement mauvais pour ce type de problème de causalité inverse**. Ils nous obligent à répondre à des questions "et si", que les économistes appellent **contre-factuelles**.

*Exemples:*

Si vous travaillez dans une banque, octroyant des crédits, vous devrez comprendre comment changer la ligne de clients modifie vos revenus. Ou, si vous travaillez dans le gouvernement local, on pourrait vous demander de trouver comment améliorer le système éducatif. **Devriez-vous donner des tablettes à chaque enfant parce que l'ère de la connaissance numérique vous le dit ?** Ou devriez-vous construire une bibliothèque à l'ancienne ?

Au cœur de ces questions, il y a une enquête causale dont nous souhaitons connaître la réponse.

Elles jouent également un rôle essentiel dans les dilemmes très personnels et chers à nos cœurs : **dois-je aller dans une école coûteuse pour réussir dans la vie (l'éducation cause-t-elle des revenus) ?** L'immigration réduit-elle mes chances de trouver un emploi (l'immigration cause-t-elle une augmentation du chômage) ?

Peu importe le domaine dans lequel vous vous trouvez. Il est très probable que vous avez eu ou aurez à répondre à un certain type de question causale. Malheureusement pour le ML, nous ne pouvons pas compter sur des prédictions de type corrélation pour les aborder.

Mais précisement pourquoi le ML nous fait défaut ?
En une phrase : **corrélation n'est pas causalité.**

[Exemples de corrélations fallacieuses](https://tylervigen.com/spurious-correlations)

## Quand la corrélation EST causalité

Intuitivement, nous savons pourquoi l'association n'est pas la causalité. Si quelqu'un vous dit que les écoles qui donnent des tablettes à leurs élèves obtiennent de meilleurs résultats que celles qui n'en donnent pas, vous pouvez rapidement signaler qu'il est probablement le cas que ces écoles avec des tablettes sont plus riches. En tant que telles, elles feraient mieux que la moyenne même sans les tablettes. À cause de cela, nous ne pouvons pas conclure que donner des tablettes aux enfants pendant les cours entraînera une augmentation de leurs performances académiques. Nous pouvons seulement dire que les tablettes à l'école sont associées à une haute performance académique, mesurée par l'ENEM (une sorte de BAC au Brésil, qui signifie Examen National du Lycée) :

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/data/"

In [ ]:
# @title
import pandas as pd
import numpy as np
from scipy.special import expit
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib import style

style.use("fivethirtyeight")

np.random.seed(123)
n = 100
tuition = np.random.normal(1000, 300, n).round()
tablet = np.random.binomial(1, expit((tuition - tuition.mean()) / tuition.std())).astype(bool)
enem_score = np.random.normal(200 - 50 * tablet + 0.7 * tuition, 200)
enem_score = (enem_score - enem_score.min()) / enem_score.max()
enem_score *= 1000

data = pd.DataFrame(dict(enem_score=enem_score, Tuition=tuition, Tablet=tablet))

In [ ]:
# @title
plt.figure(figsize=(6,8))
sns.boxplot(y="enem_score", x="Tablet", data=data).set_title('ENEM score by Tablet in Class')
plt.show()

### Formaliser l'intuition: point sur les notations

Pour aller au-delà de la simple intuition, établissons d'abord une notation. Ce sera notre langage quotidien pour parler de causalité. Considérez-le comme la langue commune que nous utiliserons.

Appelons $T_i$ la prise de traitement pour l’unité $i$.

$$
T_i =
\begin{cases}
1 & \text{si l’unité } i \text{ a reçu le traitement},\\
0 & \text{sinon}.
\end{cases}
$$

"Traitement" = Terme utilisé pour désigner une intervention dont nous voulons connaître l'effet.

## Le raisonnement contrefactuel

### Outcome potentiel et contrefactuel

Maintenant, appelons $Y_i$ la variable de résultat observée pour l'unité i.

**Le résultat est notre variable d'intérêt. Nous voulons savoir si le traitement a une influence sur celle-ci.** Dans notre exemple des tablettes, ce serait la performance académique.

C'est ici que les choses deviennent intéressantes. **Le problème fondamental de l'inférence causale est que nous ne pouvons jamais observer la même unité avec et sans traitement.** C'est comme si nous avions deux chemins divergents et que nous ne pouvions savoir ce qui se trouve devant que sur celui que nous prenons. Comme dans le poème de Robert Frost :

> Two roads diverged in a yellow wood,<br>
> And sorry I could not travel both<br>
> And be one traveler, long I stood<br>
> And looked down one as far as I could<br>
> To where it bent in the undergrowth;

Pour comprendre cela, nous parlerons beaucoup en termes de résultats **potentiels**. Ils sont potentiels parce qu'ils ne se sont pas réellement produits. Au lieu de cela, ils désignent ce qui se serait passé dans le cas où un traitement aurait été pris. Nous appelons parfois le résultat potentiel qui s'est produit, factuel, et celui qui ne s'est pas produit, **contrefactuel**.

En ce qui concerne la notation, nous utilisons un indice supplémentaire :

**$Y_{0i}$ est le résultat potentiel pour l'unité i sans le traitement.**

**$Y_{1i}$ est le résultat potentiel pour la même unité i avec le traitement.**

Pour revenir à notre exemple, $Y_{1i}$ est la performance académique de l'étudiant i s'il ou elle est dans une classe avec des tablettes. Que ce soit le cas ou non, cela n'a pas d'importance pour $Y_{1i}$. C'est le même résultat dans tous les cas. Si l'étudiant i obtient la tablette, nous pouvons observer $Y_{1i}$. Sinon, nous pouvons observer $Y_{0i}$. Notez que dans ce dernier cas, $Y_{1i}$ est toujours défini, nous ne pouvons tout simplement pas le voir. Dans ce cas, il s'agit d'un résultat potentiel contrefactuel.

### ATT et ATE

Avec les résultats potentiels, nous pouvons définir l'effet du traitement individuel :

$Y_{1i} - Y_{0i}$

Bien sûr, en raison du problème fondamental de l'inférence causale, nous ne pouvons jamais connaître l'effet du traitement individuel car nous n'observons qu'un des résultats potentiels.

Pour l'instant, concentrons-nous sur quelque chose de plus facile que d'estimer l'effet du traitement individuel. Au lieu de cela, concentrons-nous sur l'effet **moyen du traitement**, qui est défini comme suit :

$$ATE = E[Y_1 - Y_0]$$

où $E[...]$ est la valeur attendue. Une autre quantité plus facile à estimer est **l'effet moyen du traitement sur les traités**:

$$ATT = E[Y_1-Y_0|T=1]$$

Nous ne pouvons pas voir les deux résultats potentiels, mais juste pour l'argument, supposons que nous le pouvions. Prétendons que la divinité de l'inférence causale est satisfaite des nombreuses batailles statistiques que nous avons menées et nous a récompensés de pouvoirs divins pour voir les résultats alternatifs potentiels. Avec ce pouvoir, disons que nous collectons des données sur 4 écoles. Nous savons si elles ont donné des tablettes à leurs étudiants et leur score à certains tests académiques annuels. Ici, les tablettes sont le traitement, donc $T=1$ si l'école fournit des tablettes à ses élèves. $Y$ sera le score au test.

In [ ]:
# @title
pd.DataFrame(dict(
    i= [1,2,3,4],
    Y0=[500,600,800,700],
    Y1=[450,600,600,750],
    T= [0,0,1,1],
    Y= [500,600,600,750],
    TE=[-50,0,-200,50],
))

Le $ATE$ ici serait la moyenne de la dernière colonne, c'est-à-dire de l'effet du traitement :

$ATE = (-50 + 0 - 200 + 50)/4 = -50$

Cela signifierait que les tablettes ont réduit la performance académique des étudiants, en moyenne, de 50 points. Le $ATT$ ici serait la moyenne de la dernière colonne lorsque $T = 1$:

$ATT = (-200 + 50)/2 = -75$

Cela signifie que, pour les écoles qui ont été traitées, les tablettes ont réduit la performance académique des étudiants, en moyenne, de 75 points. Bien sûr, nous ne pouvons jamais savoir cela. En réalité, le tableau ci-dessus ressemblerait à ceci :

In [ ]:
# @title
pd.DataFrame(dict(
    i= [1,2,3,4],
    Y0=[500,600,np.nan,np.nan],
    Y1=[np.nan,np.nan,600,750],
    T= [0,0,1,1],
    Y= [500,600,600,750],
    TE=[np.nan,np.nan,np.nan,np.nan],
))

Vous pourriez dire que ce n'est sûrement pas idéal, mais ne puis-je pas encore prendre la moyenne des traités et la comparer à la moyenne des non-traités ? En d'autres termes, ne puis-je pas simplement faire $ATE=(600+750)/2-(500+600)/2=125$ ? Eh bien, non ! Remarquez à quel point les résultats sont différents. Vous venez de commettre le péché le plus grave en confondant association et causalité. Pour comprendre pourquoi, examinons le principal ennemi de l'inférence causale.

## Biais

Le biais est ce qui rend l'association différente de la causalité. Heureusement, cela peut être facilement compris avec notre intuition.

Revenons à notre exemple des tablettes en classe. Lorsqu'on est confronté à l'affirmation selon laquelle les écoles qui donnent des tablettes à leurs élèves obtiennent de meilleurs scores aux tests, nous pouvons la réfuter en disant que ces écoles obtiendront probablement de meilleurs scores aux tests de toute façon, même sans les tablettes. C'est parce qu'elles ont probablement plus d'argent que les autres écoles ; elles peuvent donc payer de meilleurs enseignants, se permettre de meilleures salles de classe, etc. En d'autres termes, il est probable que les écoles traitées (avec tablettes) ne sont **pas comparables** aux écoles non traitées.


Avec cela en tête, nous pouvons montrer avec des mathématiques élémentaires pourquoi l'association n'est pas la causalité. L'association est mesurée par $E[Y|T=1] - E[Y|T=0]$. Dans notre exemple, il s'agit du score moyen aux tests pour les écoles avec tablettes moins le score moyen aux tests pour celles sans tablettes. D'autre part, la causalité est mesurée par $E[Y_1 - Y_0]$.

Prenons la mesure de l'association et remplaçons les résultats observés par les résultats potentiels pour voir comment ils sont liés. Pour les traités, le résultat observé est $Y_1$. Pour les non-traités, le résultat observé est $Y_0$.

$$E[Y|T=1] - E[Y|T=0] = E[Y_1|T=1] - E[Y_0|T=0]$$

Maintenant, ajoutons et soustrayons $E[Y_0|T=1]$. Il s'agit d'un résultat contrefactuel. Il indique quel aurait été le résultat des traités, s'ils n'avaient pas reçu le traitement.

$$E[Y|T=1] - E[Y|T=0] = E[Y_1|T=1] - E[Y_0|T=0] + E[Y_0|T=1] - E[Y_0|T=1]$$

Enfin, réorganisons les termes, fusionnons certaines attentes, et voilà

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
\underset{\mathrm{ATT}}
{\Bigl(E[Y_1-Y_0 \mid T=1]\Bigr)}
+
\underset{\text{Biais de sélection}}
{\Bigl(E[Y_0 \mid T=1]-E[Y_0 \mid T=0]\Bigr)}.
$$

**Ce simple morceau de mathématiques englobe tous les problèmes que nous rencontrerons dans les questions causales.**

Décomposons-la en certaines de ses implications. Premièrement, cette équation explique pourquoi l'association n'est pas la causalité. Comme nous pouvons le voir, l'association est égale à l'effet du traitement sur les traités plus un terme de biais. Le biais est donné par la différence entre le groupe traité et le groupe de contrôle avant le traitement, au cas où aucun d'eux n'aurait reçu le traitement. Nous pouvons maintenant dire précisément pourquoi nous sommes méfiants lorsque quelqu'un nous dit que les tablettes en classe améliorent les performances académiques. Nous pensons que, dans cet exemple, $E[Y_0|T=0] < E[Y_0|T=1]$, c'est-à-dire que les écoles qui peuvent se permettre de donner des tablettes à leurs élèves sont meilleures que celles qui ne le peuvent pas, indépendamment du traitement par les tablettes.

Pourquoi cela se produit-il ?

Parce que de nombreuses choses que nous ne pouvons pas contrôler changent en même temps que le traitement. En conséquence, les écoles traitées et non traitées ne diffèrent pas seulement par les tablettes. Elles diffèrent également par le coût de la scolarité, l'emplacement, les enseignants...

In [ ]:
# @title
plt.figure(figsize=(10,6))
sns.scatterplot(x="Tuition", y="enem_score", hue="Tablet", data=data, s=70).set_title('ENEM score by Tuition Cost')
plt.show()

Maintenant que nous comprenons le problème, examinons la solution. Nous pouvons également dire ce qui serait nécessaire pour que l'association soit égale à la causalité. Si $E[Y_0|T=0] = E[Y_0|T=1]$
, alors, l'association EST LA CAUSALITÉ !

Dire que $E[Y_0|T=0]=E[Y_0|T=1]$ revient à dire que le groupe de traitement et le groupe de contrôle sont comparables avant le traitement. Autrement dit, lorsque les traités n'ont pas été traités, si nous pouvions observer leur $Y_0$, leur résultat serait le même que celui des non-traités. Mathématiquement, le terme de biais disparaîtrait :

$$E[Y|T=1]-E[Y|T=0]=E[Y_1-Y_0|T=1]=ATT$$

De plus, si les traités et les non-traités ne diffèrent que par le traitement lui-même, alors $E[Y_0|T=0]=E[Y_0|T=1]$ et nous avons que l'impact causal sur les traités est le même que pour les non-traités (parce qu'ils sont très similaires).

\begin{align}
\mathbb{E}[Y_1-Y_0 \mid T=1]
&= \mathbb{E}[Y_1 \mid T=1]
 - \mathbb{E}[Y_0 \mid T=1] \tag{1}\\
&= \mathbb{E}[Y_1 \mid T=1]
 - \mathbb{E}[Y_0 \mid T=0] \tag{2}\\
&= \mathbb{E}[Y \mid T=1]
 - \mathbb{E}[Y \mid T=0] \tag{3}
\end{align}

Dans ce cas, la **différence de moyennes DEVIENT l'effet causal**:

$E[Y|T=1]-E[Y|T=0]=ATT$

De plus, si les traités et les non-traités ne diffèrent que par le traitement lui-même, nous avons également $E[Y_1|T=0]=E[Y_1|T=1]$, c'est-à-dire que nous nous assurons que les groupes traités et de contrôle répondent de manière similaire au traitement. Maintenant, en plus d'être interchangeables avant le traitement, les traités et les non-traités sont également interchangeables après le traitement.

Dans ce cas,

$$E[Y_1-Y_0|T=1]=E[Y_1-Y_0|T=0] $$ et $$E[Y|T=1]-E[Y|T=0]=ATT=ATE$$

Comparaison de moyennes entre le groupe traité et le groupe non traité (les points bleus n'ont pas reçu le traitement):

![anatomy1.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/anatomy1.png)

Remarquez comment la différence de résultats entre les deux groupes peut avoir deux causes :

1. L'effet du traitement. L'augmentation des scores aux tests est causée par le fait de donner des tablettes aux enfants.
2. Certaines des différences dans les scores aux tests peuvent être dues aux frais de scolarité pour une meilleure éducation. Dans ce cas, les traités et les non-traités diffèrent parce que les traités ont des frais de scolarité beaucoup plus élevés. D'autres différences entre le traitement et les non-traités ne sont PAS le traitement lui-même.

L'effet du traitement individuel est la différence entre le résultat de l'unité et un autre résultat théorique que la même unité aurait si elle avait reçu le traitement alternatif. L'effet réel du traitement ne peut être obtenu que si nous avons des pouvoirs divins pour observer le résultat potentiel, comme dans la figure de gauche ci-dessous. Ce sont les résultats contrefactuels et sont indiqués en couleur claire.

![anatomy2.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/anatomy2.png)

Dans le graphique de droite, nous avons représenté le biais dont nous avons parlé précédemment. Nous obtenons le biais si nous décidons que tout le monde ne reçoit pas le traitement. Dans ce cas, il ne nous reste que le résultat potentiel $Y_0$. Ensuite, nous voyons comment les groupes traités et non traités diffèrent. S'ils diffèrent, quelque chose d'autre que le traitement cause la différence entre les traités et les non-traités. Cette chose est le biais et c'est ce qui masque l'effet réel du traitement.

Maintenant, comparez cela à une situation hypothétique où il n'y a pas de biais. Supposons que les tablettes soient attribuées au hasard aux écoles. Dans cette situation, les écoles riches et pauvres ont les mêmes chances de recevoir le traitement. Le traitement serait bien réparti sur l'ensemble du spectre des frais de scolarité.

![anatomy3.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/anatomy3.png)

Dans ce cas, la différence de résultats entre les traités et les non-traités EST l'effet causal moyen. Cela se produit parce qu'il n'y a pas d'autre source de différence entre les traités et les non-traités autre que le traitement lui-même. Toutes les différences que nous voyons doivent lui être attribuées. Une autre façon de dire cela est qu'il n'y a pas de biais.

![anatomy4.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/anatomy4.png)

Si nous décidons que tout le monde ne reçoit pas le traitement afin que nous n'observions que les $Y_0$, nous ne trouverions aucune différence entre les groupes traités et non traités.

C'est la tâche herculéenne de l'inférence causale. Il s'agit de trouver des moyens astucieux de supprimer le biais et de rendre les traités et les non-traités comparables afin que toute la différence que nous voyons ne soit que l'effet moyen du traitement. En fin de compte, l'inférence causale consiste à comprendre comment le monde fonctionne, dépouillé de toutes les illusions et mauvaises interprétations. Et maintenant que nous comprenons cela, nous pouvons avancer pour maîtriser certaines des méthodes les plus puissantes pour éliminer le biais, les armes des Braves et des Véritables, pour identifier l'effet causal.

# 02 - Expériences Contrôlées Aléatoires

## La norme d'or

$$
E[Y \mid T=1]-E[Y \mid T=0]
=
\underset{\mathrm{ATT}}
{\Bigl(E[Y_1-Y_0 \mid T=1]\Bigr)}
+
\underset{\text{Biais de sélection}}
{\Bigl(E[Y_0 \mid T=1]-E[Y_0 \mid T=0]\Bigr)}.
$$

Pour récapituler, l'association devient causalité s'il n'y a pas de biais. Il n'y aura pas de biais si $E[Y_0|T=0] = E[Y_0|T=1]$
. En d'autres termes, l'association sera la causalité si les traités et les contrôles sont égaux ou comparables, à l'exception de leur traitement. Ou, en termes plus techniques, lorsque le résultat des non-traités est égal au résultat contrefactuel des traités. Rappelez-vous que ce résultat contrefactuel est le résultat du groupe traité s'il n'avait pas reçu le traitement.

Examinons le premier outil que nous avons pour faire disparaître le biais : les **expériences randomisées**.

**Définition** : Les expériences randomisées assignent aléatoirement des individus dans une population à un groupe de traitement ou à un groupe de contrôle.

La randomisation annihile le biais en rendant les *résultats potentiels* indépendants du traitement.

$$ (Y_0, Y_1) \perp \!\!\! \perp T $$

Si le résultat est indépendant du traitement, cela n'implique-t-il pas également que le traitement n'a aucun effet ? Eh bien, oui ! Mais on parle ici de *résultats POTENTIELS*.

Le résultat potentiel est ce que le résultat aurait été sous traitement (
$Y_1$) ou sous contrôle ($Y_0$). Dans les essais randomisés, nous ne voulons pas que le résultat soit indépendant du traitement puisque nous pensons que le traitement cause le résultat. Mais nous voulons que les résultats potentiels soient indépendants du traitement.

![indep.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/indep.png)

Dire que les résultats potentiels sont indépendants du traitement revient à dire qu'ils seraient, en moyenne, les mêmes dans le groupe de traitement ou dans le groupe de contrôle. En termes plus simples, cela signifie que les groupes de traitement et de contrôle sont comparables.

Par conséquent, $(Y_0, Y_1) \perp T$ signifie que le traitement est la seule chose générant une différence entre le résultat dans le groupe traité et dans le groupe de contrôle.

Formalisation:

L'indépendance implique précisément que

$E[Y_0|T=0] = E[Y_0|T=1] = E[Y_0]$

Ce qui fait que

$E[Y|T=1]-E[Y|T=0]=E[Y_1-Y_0]=ATE$

**Ainsi, utiliser la randomisation donne à la simple différence de moyenne entre le traitement et le contrôle la qualité d' "effet de traitement".**

## Dans une Ecole très, très lointaine
### Illustrer les expériences randomisées avec la pandémie du Covid-19

En 2020 la pandémie du Coronavirus force les écoles à s'adapter aux mesures de distanciation sociales. Beaucoup d'école ferment et déposent leurs cours en ligne.

Quatre mois après le début de la crise, beaucoup se demandent si les changements introduits pourraient être maintenus. Il ne fait aucun doute que l'apprentissage en ligne présente des avantages. Il est moins cher car il permet d'économiser sur l'immobilier et le transport. Il peut également être plus numérique, tirant parti de contenus de classe mondiale du monde entier, et pas seulement d'un ensemble fixe d'enseignants. **Malgré tout cela, nous devons encore répondre à la question de savoir si l'apprentissage en ligne a un impact négatif ou positif sur la performance académique des étudiants.**

Une façon de répondre à cette question est de prendre des étudiants de écoles qui donnent principalement des cours en ligne et de les comparer avec des étudiants de écoles qui donnent des cours en présentiel. Comme nous le savons maintenant, ce n'est pas la meilleure approche. Il se pourrait que les écoles en ligne attirent uniquement les étudiants bien disciplinés qui réussissent mieux que la moyenne, même si le cours était en présentiel. Dans ce cas, nous aurions un biais positif, où les traités sont académiquement meilleurs que les non-traités : $E[Y_0|T=1] > E[Y_0|T=0]$.

Donc, bien que nous puissions faire des comparaisons simples, ce ne serait pas convaincant. D'une manière ou d'une autre, nous ne pourrions jamais être sûrs qu'il n'y avait pas de biais caché qui masquait notre effet causal.

![lurking_bias.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/lurking_bias.png)

Pour résoudre ce problème, nous devons rendre les traités et les non-traités comparables $E[Y_0|T=1] = E[Y_0|T=0]$. Une façon de forcer cela est d'assigner aléatoirement les cours en ligne et en présentiel aux étudiants. Si nous parvenions à le faire, les traités et les non-traités seraient, en moyenne, les mêmes, à l'exception du traitement qu'ils reçoivent.

Heureusement, certains économistes ont fait cela pour nous. Ils ont randomisé les cours de sorte que certains étudiants ont été assignés à avoir des cours en présentiel, d'autres à avoir uniquement des cours en ligne, et un troisième groupe à avoir un format mixte de cours en ligne et en présentiel. Ils ont collecté des données sur un examen standard à la fin du semestre.

In [ ]:
data = pd.read_csv(BASE + "online_classroom.csv")

print(data.shape)
data.head()

Nous pouvons voir que nous avons 323 échantillons. Ce n’est pas exactement du big data, mais c’est quelque chose avec lequel nous pouvons travailler. Pour estimer l’effet causal, nous pouvons simplement calculer la moyenne des scores pour chacun des groupes de traitement.

In [ ]:
(data
 .assign(class_format = np.select(
     [data["format_ol"].astype(bool), data["format_blended"].astype(bool)],
     ["online", "blended"],
     default="face_to_face"
 ))
 .groupby(["class_format"])
 .mean())

Oui. C'est aussi simple que cela. Nous pouvons voir que les cours en présentiel obtiennent un score moyen de 78,54, tandis que les cours en ligne obtiennent un score moyen de 73,63. Pas de très bonnes nouvelles pour les partisans de l'apprentissage en ligne. L'ATE pour un cours en ligne est donc de -4,91. Cela signifie que les cours en ligne font en sorte que les étudiants obtiennent environ 5 points de moins, en moyenne. C'est tout. Vous n'avez pas à vous inquiéter que les cours en ligne puissent avoir des étudiants plus pauvres qui ne peuvent pas se permettre des cours en présentiel ou, d'ailleurs, vous n'avez pas à vous inquiéter que les étudiants des différents traitements soient différents de quelque manière que ce soit, autre que le traitement qu'ils ont reçu. Par conception, l'expérience randomisée est faite pour éliminer ces différences.

Pour cette raison, une bonne vérification de la validité de la randomisation (ou si vous examinez les bonnes données) consiste à vérifier si les traités sont égaux aux non-traités dans les variables pré-traitement. Nos données contiennent des informations sur le genre et l'ethnicité pour voir s'ils sont similaires entre les groupes. Nous pouvons dire qu'ils semblent assez similaires pour les variables genre, asiatique, hispanique et blanc. La variable noir, cependant, semble un peu différente. Cela attire l'attention sur ce qui se passe avec un petit ensemble de données. Même sous randomisation, il se peut que, par hasard, un groupe soit différent d'un autre. Dans les grands échantillons, cette différence tend à disparaître.

## L'expérience idéale

Les expériences randomisées ou les essais contrôlés randomisés (ECR) sont le moyen le plus fiable d'obtenir des effets causals. C'est une technique simple et extrêmement convaincante. Elle est si puissante que la plupart des pays l'exigent pour démontrer l'efficacité de nouveaux médicaments. **Pensez-y de cette façon, si nous le pouvions, les ECR seraient tout ce que nous ferions pour découvrir la causalité. Un ECR bien conçu est le rêve de tout scientifique.**

![science_dream.png](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/science_dream.png)

Malheureusement, ils tendent à être soit très chers, soit tout simplement contraires à l'éthique. Parfois, nous ne pouvons tout simplement pas contrôler le mécanisme d'attribution. Imaginez-vous en tant que médecin essayant d'estimer l'effet du tabagisme pendant la grossesse sur le poids du bébé à la naissance. Vous ne pouvez pas simplement forcer une portion aléatoire de mamans à fumer pendant la grossesse. Ou disons que vous travaillez pour une grande banque et que vous devez estimer l'impact de la ligne de crédit sur la fidélisation des clients. Il serait trop coûteux de donner des lignes de crédit aléatoires à vos clients. Ou que vous voulez comprendre l'impact de l'augmentation du salaire minimum sur le chômage. Vous ne pouvez pas simplement assigner des pays à avoir un salaire minimum ou un autre. Vous voyez le problème.

Nous verrons plus tard comment réduire le coût de la randomisation en utilisant la randomisation conditionnelle, mais il n'y a rien que nous puissions faire à propos des expériences contraires à l'éthique ou irréalisables. Néanmoins, chaque fois que nous traitons des questions causales, il vaut la peine de penser à l'expérience idéale. **Demandez-vous toujours, si vous le pouviez, quelle serait l'expérience parfaite que vous mèneriez pour découvrir cet effet causal ?** Cela tend à **éclairer la manière** dont nous pouvons découvrir l'effet causal **même sans** l'expérience idéale.

## Le mécanisme d'attribution

Dans une expérience randomisée, le mécanisme qui attribue les unités à un traitement ou à un autre est, eh bien, aléatoire. Comme nous le verrons plus tard, toutes les techniques d'inférence causale essaieront d'une manière ou d'une autre d'identifier les mécanismes d'attribution des traitements. Lorsque nous savons avec certitude comment ce mécanisme se comporte, l'inférence causale sera beaucoup plus confiante, même si le mécanisme d'attribution n'est pas aléatoire.

Malheureusement, le mécanisme d'attribution ne peut pas être découvert en regardant simplement les données. Par exemple, si vous avez un ensemble de données où l'enseignement supérieur est corrélé avec la richesse, vous ne pouvez pas savoir avec certitude lequel a causé l'autre en regardant simplement les données. Vous devrez utiliser vos connaissances sur le fonctionnement du monde pour argumenter en faveur d'un mécanisme d'attribution plausible : est-ce que les écoles éduquent les gens, les rendant plus productifs et les menant à des emplois mieux rémunérés. Ou, si vous êtes pessimiste à propos de l'éducation, vous pouvez dire que les écoles ne font rien pour augmenter la productivité, et que c'est juste une corrélation fallacieuse parce que seules les familles riches peuvent se permettre d'envoyer un enfant à un diplôme supérieur.

Dans les questions causales, nous pouvons généralement argumenter dans les deux sens : que X cause Y, ou qu'il s'agit d'une troisième variable Z qui cause à la fois X et Y, et donc la corrélation entre X et Y n'est que fallacieuse. Pour cette raison, connaître le mécanisme d'attribution conduit à une réponse causale beaucoup plus convaincante. C'est aussi ce qui rend l'inférence causale si excitante. Alors que le machine learning appliqué consiste généralement à appuyer sur quelques boutons dans le bon ordre, l'inférence causale appliquée vous oblige à réfléchir sérieusement au mécanisme générant ces données.

## Et l'AB testing dans tout ça ?

Vous l'aurez remarqué on a jusqu'à présent seulement mentionné les expérimentations contrôlées aléatoires, sans jamais mentionner A/B testing. La première est une nomenclature empruntée au monde académique, tandis-que la deuxième nous vient de l'industrie et du commerce. Les deux méthodes servent a évaluer si votre hypothèse est correcte ou non, utiliser les données pour mesurer l'efficacité des interventions et tenter de comprendre les relations de causalité. Mais alors quelles similtudes et différences entre les deux ?

![marketers_AB_test.jpg](https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc5_abtest/img/marketers_ab_test.jpg)

## Définir pour différencier

L'AB testing propose deux versions d'un produit ou d'un service pour en comparer la performance. Ils sont proposé à une audience sélectionné de façon aléatoire (sample). Une comparaison statistique informée par les données collectées de cette façon doit éclairer votre décision au regard des indicateurs choisis : vente, clics, abonnements, etc.

En ce sens que les deux méthodes proposent de tester une intervention sur deux groupes similaires en utilisant la randomisation, l'AB testing est une sorte de petite et rapide RCT.

Mais l'AB testing est a appréhender comme un sous-ensemble de l'ensemble des RCT. Il est en pratique utilisé sur des périodes de temps plus courte, pour évaluer des interventions peu complexe et dans un contexte marketing/commerce numérique (mails, display de pubs, recommandations, etc).

Néanmoins, **l'inférence causale reste au coeur des préoccupations des deux méthodes**.